<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 2.1: 你的第一个 Chisel 模块
**上一节: [Scala 简介](1_intro_to_scala.ipynb)**<br>
**下一节: [组合逻辑](2.2_comb_logic.ipynb)**

## 动机
既然你已经熟悉了 Scala，让我们开始设计一些硬件吧！Chisel 代表 **C**onstructing **H**ardware **I**n a **S**cala **E**mbedded **L**anguage（在 Scala 嵌入式语言中构建硬件）。这意味着它是 Scala 中的一个领域特定语言（DSL），允许你在同一代码中同时利用 Scala 和 Chisel 编程。理解哪些代码是 "Scala" 代码，哪些是 "Chisel" 代码很重要，但我们稍后会详细讨论。现在，将 Chisel 和第 2 模块中的代码视为编写 Verilog 的更好方式。本模块将向你展示一个完整的 Chisel `Module` 和测试器。现在只需了解其要点，稍后你会看到更多示例。

## 设置
以下单元格下载 Chisel 所需的依赖项。你将在所有未来的笔记本中看到它。**现在运行此单元格**。

In [1]:
val path = System.getProperty("user.dir") + "/source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

path: String = "/home/zhaochanglong/Documents/chisel-bootcamp/source/load-ivy.sc"

如上一模块所述，这些语句是导入 Chisel 所必需的。**现在运行此单元格**，然后再运行任何未来的代码块。

In [2]:
import chisel3._
import chisel3.util._
import chisel3.tester._
import chisel3.tester.RawTester.test
import dotvisualizer._

import chisel3._

import chisel3.util._

import chisel3.tester._

import chisel3.tester.RawTester.test

import dotvisualizer._

---
# 你的第一个模块
本节将展示你的第一个硬件模块、一个测试用例以及如何运行它。其中包含许多你可能不理解的内容，这没关系。我们希望你能了解大致轮廓，以便可以不断回到这个完整且可工作的示例来巩固所学知识。

<span style="color:blue">**示例：一个模块**</span><br>
与 Verilog 类似，我们可以在 Chisel 中声明模块定义。以下示例是一个 Chisel `Module`，`Passthrough`，它有一个 4 位输入 `in` 和一个 4 位输出 `out`。该模块组合连接 `in` 和 `out`，因此 `in` 驱动 `out`。

In [3]:
// Chisel Code: Declare a new module definition
class Passthrough extends Module {
  val io = IO(new Bundle {
    val in = Input(UInt(4.W))
    val out = Output(UInt(4.W))
  })
  io.out := io.in
}

defined class Passthrough

这里有很多内容！以下解释了如何从我们描述的硬件的角度来理解每一行。

```scala
class Passthrough extends Module {
```
我们声明一个名为 `Passthrough` 的新模块。`Module` 是一个内置的 Chisel 类，所有硬件模块都必须扩展它。

```scala 
val io = IO(...)
```
我们在一个特殊的 `io` `val` 中声明所有输入和输出端口。它必须命名为 `io`，并且是一个 `IO` 对象或实例，需要 `IO(_instantiated_bundle_)` 形式的内容。

```scala
new Bundle {
    val in = Input(...)
    val out = Output(...)
}
```
我们声明一个新的硬件结构类型（Bundle），其中包含一些命名信号 `in` 和 `out`，方向分别为 Input 和 Output。

```scala
UInt(4.W)
```
我们声明信号的硬件类型。在这种情况下，它是一个宽度为 4 的无符号整数。

```scala
io.out := io.in
```
我们将输入端口连接到输出端口，使得 `io.in` *驱动* `io.out`。请注意，`:=` 运算符是一个 ***Chisel*** 运算符，表示右侧信号驱动左侧信号。它是一个有向运算符。

硬件构造语言（HCL）的巧妙之处在于，我们可以使用底层编程语言作为脚本语言。例如，在声明 Chisel 模块后，我们使用 Scala 调用 Chisel 编译器将 Chisel `Passthrough` 转换为 Verilog `Passthrough`。这个过程称为 ***elaboration***（细化）。

In [ ]:
// Scala Code: Elaborate our Chisel design by translating it to Verilog
// Don't worry about understanding this code; it is very complicated Scala
println(getVerilog(new Passthrough))

<span style="color:blue">**示例：模块生成器**</span><br>
如果我们将学到的 Scala 知识应用到这个例子中，我们可以看到 Chisel 模块是作为 Scala 类实现的。就像任何其他 Scala 类一样，我们可以让 Chisel 模块接受一些构造参数。在这种情况下，我们创建一个新类 `PassthroughGenerator`，它将接受一个整数 `width`，该参数决定其输入和输出端口的宽度：

In [ ]:
// Chisel Code, but pass in a parameter to set widths of ports
class PassthroughGenerator(width: Int) extends Module { 
  val io = IO(new Bundle {
    val in = Input(UInt(width.W))
    val out = Output(UInt(width.W))
  })
  io.out := io.in
}

// Let's now generate modules with different widths
println(getVerilog(new PassthroughGenerator(10)))
println(getVerilog(new PassthroughGenerator(20)))

请注意，生成的 Verilog 根据分配给 `width` 参数的值对输入/输出使用不同的位宽。让我们深入了解这是如何工作的。因为 Chisel 模块是普通的 Scala 类，我们可以利用 Scala 类构造函数的强大功能来参数化我们设计的细化过程。

你可能会注意到，这种参数化是由 *Scala* 实现的，而不是 *Chisel*；Chisel 没有额外的参数化 API，但设计者可以简单地利用 Scala 特性来参数化他/她的设计。

因为 `PassthroughGenerator` 不再描述单个模块，而是描述了一个由 `width` 参数化的模块家族，我们将这个 `Passthrough` 称为 ***生成器***。

---
# 测试你的硬件

任何硬件模块或生成器都不应该没有测试器。Chisel 具有内置的测试功能，你将在整个训练营中探索这些功能。以下示例是一个 Chisel 测试框架，它将值传递给 `Passthrough` 实例的输入端口 `in`，并检查输出端口 `out` 上是否看到相同的值。

<span style="color:blue">**示例：测试器**</span><br>
这里有一些高级的 Scala 代码。但是，除了 `poke` 和 `expect` 命令外，你不需要理解其他任何内容。你可以将代码的其余部分视为编写这些简单测试的样板代码。

In [ ]:
// Scala Code: `test` runs the unit test. 
// test takes a user Module and has a code block that applies pokes and expects to the 
// circuit under test (c)
test(new Passthrough()) { c =>
    c.io.in.poke(0.U)     // Set our input to value 0
    c.io.out.expect(0.U)  // Assert that the output correctly has 0
    c.io.in.poke(1.U)     // Set our input to value 1
    c.io.out.expect(1.U)  // Assert that the output correctly has 1
    c.io.in.poke(2.U)     // Set our input to value 2
    c.io.out.expect(2.U)  // Assert that the output correctly has 2
}
println("SUCCESS!!") // Scala Code: if we get here, our tests passed!


发生了什么？测试接受一个 `Passthrough` 模块，为模块的输入分配值，并检查其输出。要设置输入，我们调用 `poke`。要检查输出，我们调用 `expect`。如果我们不想将输出与预期值进行比较（无断言），我们可以改为 `peek` 输出。

如果所有 `expect` 语句都为真，那么我们的样板代码将返回通过。

>注意：`poke` 和 `expect` 使用 chisel 硬件字面量表示法。这两个操作都期望正确类型的字面量。
如果对 `UInt()` 进行 `poke`，你必须提供 `UInt` 字面量（例如：`c.io.in.poke(10.U)`），同样，如果输入是 `Bool()`，`poke` 将期望 `true.B` 或 `false.B`。



<span style="color:red">**练习：编写你自己的测试器**</span><br>
编写并执行两个测试，一个测试宽度为 10 的 `PassthroughGenerator`，第二个测试宽度为 20 的 `PassthroughGenerator`。为每个测试至少检查两个值：零和指定宽度支持的最大值。请注意，三个问号在 Scala 中有特殊含义。你可能会在这些训练营练习中经常看到它。运行带有 `???` 的代码将产生 `NotImplementedError`。用你自己的代码替换 `???`。

In [ ]:
// Test with width 10

test(???) { c =>
    ???
}

// Test with width 20

test(???) { c =>
    ???
}

println("SUCCESS!!") // Scala Code: if we get here, our tests passed!

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-1" />
<label for="check-1"><strong>解决方案</strong> (点击切换显示)</label>
<article>
<pre style="background-color:#f7f7f7">
test(new PassthroughGenerator(10)) { c =>
    c.io.in.poke(0.U)
    c.io.out.expect(0.U)
    c.io.in.poke(1023.U)
    c.io.out.expect(1023.U)
}

test(new PassthroughGenerator(20)) { c =>
    c.io.in.poke(0.U)
    c.io.out.expect(0.U)
    c.io.in.poke(1048575.U)
    c.io.out.expect(1048575.U)
}

</pre></article></div></section></div>

---
# 查看生成的 Verilog/FIRRTL

如果你在理解生成的硬件方面遇到困难，并且熟悉阅读结构性 Verilog 和/或 FIRRTL（Chisel 的中间表示，类似于仅用于综合的 Verilog 子集），那么你可以尝试查看生成的 Verilog 以了解 Chisel 执行的结果。

以下是生成 Verilog（你已经见过）和 FIRRTL 的示例。

In [ ]:
// Viewing the Verilog for debugging
println(getVerilog(new Passthrough))

In [ ]:
// Viewing the firrtl for debugging
println(getFirrtl(new Passthrough))

---
# 完成！

[返回顶部。](#top)

## <span style="color:red"> 附录：关于 "printf" 调试的说明</span>
[使用打印语句进行调试](https://stackoverflow.com/a/189570) 并不总是最好的调试方式，但通常是查看某些东西未按预期工作时发生了什么的第一步。
因为 Chisel 生成器是生成硬件的程序，所以关于打印生成器和电路状态有一些额外的细微差别。
记住你的打印语句何时执行以及正在打印什么很重要。
你可能想要打印的三种常见场景有一些重要区别：
* Chisel 生成器在电路生成期间打印
* 电路在电路仿真期间打印
* 测试器在测试期间打印

`println` 是一个内置的 Scala 函数，用于打印到控制台。它 **不能** 用于在电路仿真期间打印，因为生成的电路是 FIRRTL 或 Verilog，而不是 Scala。

以下代码块显示了不同风格的打印。

In [ ]:
class PrintingModule extends Module {
    val io = IO(new Bundle {
        val in = Input(UInt(4.W))
        val out = Output(UInt(4.W))
    })
    io.out := io.in

    printf("Print during simulation: Input is %d\n", io.in)
    // chisel printf has its own string interpolator too
    printf(p"Print during simulation: IO is $io\n")

    println(s"Print during generation: Input is ${io.in}")
}

test(new PrintingModule ) { c =>
    c.io.in.poke(3.U)
    c.clock.step(5) // circuit will print
    
    println(s"Print during testing: Input is ${c.io.in.peek()}")
}